# 03. Residential scope + DNSP/manufacturer key remapping

Takes `ami_meter_clean` (notebook 01's output) and produces two further-
filtered/relabelled tables:

* **`ami_meter_residential`** -- `ami_meter_clean`, with every site whose
  `ac_capacity_kw` exceeds a residential ceiling (30kW, see
  `ami_filter.RESIDENTIAL_MAX_AC_CAPACITY_KW`) dropped entirely. A site with
  no PV capacity on record (e.g. load-only) is **kept** -- there is nothing
  to measure against the ceiling, so it is not evidence of being oversized.
* **`ami_site_metadata_residential`** -- `ami_site_metadata`, scoped to the
  same residential sites, with `dnsp_name`/`manufacturer` **replaced** by the
  anonymised IDs from the local DNSP/OEM key-mapping CSVs (held outside this
  repository -- see the path constant in Section 1). A value found in the
  data but absent from a mapping CSV keeps its original label rather than
  becoming missing, and is reported explicitly so a gap in the mapping file
  is never silently swallowed.

**Site scope, important** (same rule notebook 02 established): every
site-level statistic here is scoped to sites actually present in
`ami_meter_clean`, not the broader `ami_site_metadata` table.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import duckdb
import pandas as pd

from bms_sa_review.ami_analysis.lib import ami_filter as Filter
from bms_sa_review.synthetic_ami_creation.config import ami_config as Config

ARTEFACT_DIR = REPO_ROOT / "bms_sa_review" / "ami_analysis" / "artefacts"
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()


## 1. Load the DNSP/OEM key mappings

These CSVs are held locally, outside the repository (they are not
regenerable from the AMI extract and aren't something to commit to git).
Override `CICCADA_KEY_MAPPING_DIR` if your copy lives somewhere else.


In [ ]:
KEY_MAPPING_DIR = Path(os.environ.get(
    "CICCADA_KEY_MAPPING_DIR",
    r"C:\Users\z3553082\OneDrive - UNSW\Documents\CICCADA - Local key mapping",
))

dnsp_mapping_frame = pd.read_csv(KEY_MAPPING_DIR / "DNSP.csv")
oem_mapping_frame = pd.read_csv(KEY_MAPPING_DIR / "OEM.csv")

dnsp_mapping = Filter.build_key_mapping(dnsp_mapping_frame, key_column="dnsp", value_column="dnsp_id")
oem_mapping = Filter.build_key_mapping(oem_mapping_frame, key_column="manufacturer", value_column="m_id")

print(f"Loaded {len(dnsp_mapping):,} DNSP keys and {len(oem_mapping):,} manufacturer keys from {KEY_MAPPING_DIR}")


## 2. Site metadata, scoped to `ami_meter_clean`

Same `ami_site_metadata_clean` construction as notebook 02: only sites that
actually appear in `ami_meter_clean` are in scope for anything below.


In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter_clean AS
    SELECT * FROM read_parquet(
        '{Config.store_path("ami_meter_clean").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")
con.sql(f"""
    CREATE OR REPLACE VIEW ami_site_metadata AS
    SELECT * FROM read_parquet('{Config.store_path("ami_site_metadata").as_posix()}')
""")
con.sql("""
    CREATE OR REPLACE VIEW ami_site_metadata_clean AS
    SELECT m.*
    FROM ami_site_metadata m
    INNER JOIN (SELECT DISTINCT site_id FROM ami_meter_clean) c USING (site_id)
""")

site_meta = con.sql("SELECT * FROM ami_site_metadata_clean").df()
print(f"{len(site_meta):,} sites in ami_meter_clean scope.")


## 3. Residential-scope filter

Drop sites whose `ac_capacity_kw` exceeds the residential ceiling
(`ami_filter.RESIDENTIAL_MAX_AC_CAPACITY_KW` = 30kW). A null capacity (no PV
metadata at all -- e.g. a load-only site) is kept, not dropped: see
`ami_filter.flag_oversized_capacity`'s docstring for why.


In [ ]:
oversized = Filter.flag_oversized_capacity(site_meta)
residential_site_meta = site_meta.loc[~oversized].copy()
residential_site_ids = set(residential_site_meta.site_id)

n_total, n_oversized, n_kept = len(site_meta), int(oversized.sum()), len(residential_site_meta)
print(f"{n_total:,} sites in scope -> {n_oversized:,} dropped (ac_capacity_kw > "
      f"{Filter.RESIDENTIAL_MAX_AC_CAPACITY_KW:.0f}kW, {n_oversized / n_total:.2%}) "
      f"-> {n_kept:,} residential sites kept.")

site_meta.loc[oversized, ["site_id", "ac_capacity_kw", "dnsp_name", "manufacturer"]].sort_values(
    "ac_capacity_kw", ascending=False
).head(20)


## 4. Replace DNSP/manufacturer labels with the local key IDs

`dnsp_name`/`manufacturer` are overwritten in place with the mapped ID
(`dnsp_id`/`m_id`). Any value present in the data but absent from the
mapping CSV keeps its original label -- it is NOT silently dropped or
turned into a missing value -- and is printed below so a real mapping gap
is visible rather than an unnoticed pass-through.


In [ ]:
residential_site_meta["dnsp_name"], dnsp_unmapped = Filter.apply_key_mapping(
    residential_site_meta["dnsp_name"], dnsp_mapping,
)
residential_site_meta["manufacturer"], oem_unmapped = Filter.apply_key_mapping(
    residential_site_meta["manufacturer"], oem_mapping,
)

if dnsp_unmapped:
    print(f"WARNING -- {len(dnsp_unmapped)} DNSP name(s) with no entry in DNSP.csv "
          f"(kept as their original label): {dnsp_unmapped}")
else:
    print("Every DNSP name in scope was found in DNSP.csv.")

if oem_unmapped:
    print(f"WARNING -- {len(oem_unmapped)} manufacturer name(s) with no entry in OEM.csv "
          f"(kept as their original label): {oem_unmapped}")
else:
    print("Every manufacturer name in scope was found in OEM.csv.")


## 5. Write `ami_site_metadata_residential`


In [ ]:
SITE_META_OUT_PATH = Config.store_path("ami_site_metadata_residential")
SITE_META_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
residential_site_meta.to_parquet(SITE_META_OUT_PATH, compression="zstd", index=False)
print(f"Wrote {len(residential_site_meta):,} rows to {SITE_META_OUT_PATH}")


## 6. Filter `ami_meter_clean` to residential sites, one month at a time

Mirrors notebook 01's month-by-month processing pattern rather than loading
the full table into memory at once.


In [ ]:
RESIDENTIAL_MONTH_DIR = Config.store_path("ami_meter_residential")
RESIDENTIAL_MONTH_DIR.mkdir(parents=True, exist_ok=True)

site_ids_sql = ",".join(str(s) for s in sorted(residential_site_ids))

MONTHS = con.sql("SELECT DISTINCT year, month FROM ami_meter_clean ORDER BY year, month").df()
MONTHS_LIST = list(MONTHS.itertuples(index=False, name=None))
print(f"{len(MONTHS_LIST):,} (year, month) partitions to process.")

for year, month in MONTHS_LIST:
    month_frame = con.sql(f"""
        SELECT * FROM ami_meter_clean
        WHERE year = {year} AND month = {month} AND site_id IN ({site_ids_sql})
    """).df()
    n_before = con.sql(f"""
        SELECT count(*) FROM ami_meter_clean WHERE year = {year} AND month = {month}
    """).fetchone()[0]

    part_dir = RESIDENTIAL_MONTH_DIR / f"dt_month={year:04d}-{month:02d}"
    part_dir.mkdir(parents=True, exist_ok=True)
    month_frame.to_parquet(part_dir / "part.parquet", compression="zstd", index=False)

    print(f"{year}-{month:02d}: {n_before:,} rows in, {len(month_frame):,} written "
          f"({(n_before - len(month_frame)) / n_before:.2%} dropped by residential scope).")


## 7. Sanity check

Re-open both output tables from disk (not the in-memory frames above) and
confirm the residential ceiling and site scope both hold.


In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter_residential AS
    SELECT * FROM read_parquet(
        '{RESIDENTIAL_MONTH_DIR.as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")
con.sql(f"""
    CREATE OR REPLACE VIEW ami_site_metadata_residential AS
    SELECT * FROM read_parquet('{SITE_META_OUT_PATH.as_posix()}')
""")

checks = con.sql("""
    SELECT
      (SELECT count(*) FROM ami_meter_residential) AS n_rows,
      (SELECT count(DISTINCT site_id) FROM ami_meter_residential) AS n_sites_in_meter,
      (SELECT count(*) FROM ami_site_metadata_residential) AS n_sites_in_metadata,
      (SELECT count(*) FROM ami_site_metadata_residential WHERE ac_capacity_kw > 30) AS n_oversized_remaining,
      (SELECT count(*) FROM ami_meter_residential m
         WHERE NOT EXISTS (SELECT 1 FROM ami_site_metadata_residential s WHERE s.site_id = m.site_id)
      ) AS n_meter_rows_with_no_residential_metadata
""").df()

assert checks.n_oversized_remaining.iloc[0] == 0, "an oversized site survived the filter -- investigate"
assert checks.n_meter_rows_with_no_residential_metadata.iloc[0] == 0, (
    "ami_meter_residential contains a site that isn't in ami_site_metadata_residential -- investigate"
)
print("Confirmed: no oversized sites remain, and every ami_meter_residential site "
      "has matching residential metadata.")
checks


## 8. Export limit

Same stat as notebook 02, Section 8, re-scoped to the residential fleet.
Self-contained: re-creates the `ami_site_metadata_residential` view itself
(cheap -- just a file open) rather than relying on Section 7 having already
run, since this section doesn't need anything else Section 7 checks.


In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW ami_site_metadata_residential AS
    SELECT * FROM read_parquet('{Config.store_path("ami_site_metadata_residential").as_posix()}')
""")

export_limit_stats = con.sql("""
    SELECT count(*) AS n_total, sum(CASE WHEN export_limit_kw IS NOT NULL THEN 1 ELSE 0 END) AS n_export_limit
    FROM ami_site_metadata_residential
""").df()
n_total = int(export_limit_stats.n_total.iloc[0])
n_export_limit = int(export_limit_stats.n_export_limit.iloc[0])
print(f"{n_export_limit:,} of {n_total:,} residential sites have a non-null export_limit_kw.")
